### 1.问题
只有bash时，所有操作都走shell。cat截断不可预测，sed遇到特殊字符就崩，每次bash调用都是不受约束的安全面。专用工具（read_file，write_file）可以在工具层面做路径沙箱。
关键步骤：做到加工具不需要改循环
### 2.解决工作原理
1. 每个工具都有一个处理函数。路径沙箱防止逃逸出工作区。
```python
def safe_path(p: str) -> Path:
    path = (WORKDIR / p).resolve()
    # 进行其他安全检查
    if not path.is_relative_to(WORKDIR):
        raise ValueError(f"Path escapes workspace: {p}")
    return path

def run_read(path: str, limit: int = None) -> str:
    text = safe_path(path).read_text()
    lines = text.splitlines()
    if limit and limit < len(lines):
        lines = lines[:limit]
    return "\n".join(lines)[:50000]
```
2. dispatch map将工具名映射到处理函数。
```python
TOOL_HANDLERS = {
    "bash":    lambda **kw: run_bash(kw["command"]),
    "read_file": lambda **kw: run_read(kw["path"], kw["limit"]),
    "write_file": lambda **kw: run_write(kw["path"], kw["content"]),
    "edit_file": lambda **kw: run_edit(kw["path"], kw["old_text"], kw["new_text"]),
}
```
3. 循环中按名称查找处理函数。循环体本身和01完全一致。
```python
for tool_call in message.tool_calls:
    handler = TOOL_HANDLERS.get(tool_call.function.name)
    output = handler(**tool_call.function.arguments) if handler else f"Unknown tool: {tool_call.function.name}"
    tool_results.append({
        "tool_call_id": tool_call.id,
        "role": "tool",
        "name": "bash",
        "content": output,
    })
```
加工具 = 加 handler + 加 schema。循环永远不变。

In [ ]:
import os
import subprocess
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(base_url=os.getenv("OPENAI_API_BASE"))

WORKDIR = Path.cwd()

SYSTEM_PROMPT = f"You are a coding agent at {WORKDIR}. Use tools to solve tasks. Act, do not explain."

def safe_path(p: str) -> Path:
    path = (WORKDIR / p).resolve()
    if not path.is_relative_to(WORKDIR):
        raise ValueError(f"Path escapes workspace: {p}")
    return path

def run_bash(command: str) -> str:
    dangerous = ["rm -rf /", "sudo", "shutdown", "reboot", "> /dev/"]
    if any(d in command for d in dangerous):
        return "Error: Dangerous command blocked"
    try:
        r = subprocess.run(command, shell=True, cwd=os.getcwd(),
                           capture_output=True, text=True, timeout=120)
        out = (r.stdout + r.stderr).strip()
        return out[:50000] if out else "(no output)"
    except subprocess.TimeoutExpired:
        return "Error: Timeout (120s)"

def run_read(path: str, limit: int = None) -> str:
    try:
        text = safe_path(path).read_text()
        lines = text.splitlines()
        if limit and limit < len(lines):
            lines = lines[:limit] + [f"...({len(lines) - limit} more lines)"]
        return "\n".join(lines)[:50000]
    except Exception as e:
        return f"Error: {e}"

def run_write(path: str, content: str) -> str:
    try:
        fp = safe_path(path)
        fp.parent.mkdir(parents=True, exist_ok=True)
        fp.write_text(content)
        return f"Wrote {len(content)} bytes to {path}"
    except Exception as e:
        return f"Error: {e}"

def run_edit(path: str, old_text: str, new_text: str) -> str:
    try:
        fp = safe_path(path)
        content = fp.read_text()
        if old_text not in content:
            return f"Error: {old_text} not found in {path}"
        fp.write_text(content.replace(old_text, new_text, 1))
        return f"Edited {path}"
    except Exception as e:
        return f"Error: {e}"

TOOL_HANDLERS = {
    "bash": lambda **kw: run_bash(kw["command"]),
    "read": lambda **kw: run_read(kw["path"]),
    "write": lambda **kw: run_write(kw["path"], kw["content"]),
    "edit": lambda **kw: run_edit(kw["path"], kw["old_text"], kw["new_text"]),
}

TOOLS = [
    
]
       
